<a href="https://colab.research.google.com/github/Pratikshapawar-star/python-internship-practice-code/blob/main/Day10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
import joblib
from sklearn.svm import SVC

In [ ]:
df=pd.read_csv('/content/Churn_Modelling.csv')

FileNotFoundError: [Errno 2] No such file or directory: '/content/Churn_Modelling.csv'

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
df.info()

In [ ]:
df['Exited'].value_counts()

In [ ]:
df['Age'].fillna(df['Age'].mean(),inplace=True)
df['Gender'].fillna(df['Gender'].mode()[0],inplace=True)
df['Tenure'].fillna(df['Tenure'].mode()[0],inplace=True)
df['Balance'].fillna(df['Balance'].mean(),inplace=True)
df['HasCrCard'].fillna(df['HasCrCard'].mode()[0],inplace=True)
df['Exited'].fillna(df['Exited'].mode()[0],inplace=True)
df['EstimatedSalary'].fillna(df['EstimatedSalary'].mean(),inplace=True)

In [ ]:
x=df.drop('Exited',axis=1)
y=df['Exited']

In [ ]:
plt.bar(df['Geography'].value_counts().index,df['Geography'].value_counts() )
plt.show()

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [ ]:
max_credit_score = df['CreditScore'].max()
print(max_credit_score)

In [ ]:
min_credit_score = df['CreditScore'].min()
print(min_credit_score)

In [ ]:
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [ ]:
model1=LogisticRegression()
model1.fit(x_train,y_train)

In [ ]:
y_pred1=model1.predict(x_test)

In [ ]:
accuracy_score(y_test,y_pred1)

In [ ]:
conf_matrix=confusion_matrix(y_test,y_pred1)
sns.heatmap(conf_matrix,annot=True,fmt='d',cmap='Blues')
xticklabels=['Not Effective','Effective']
yticklabels=['Not Effective','Effective']
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
rf_model3=RandomForestClassifier(n_estimators=100,random_state=42)
rf_model3.fit(x_train,y_train)

In [ ]:
y_pred_rf3=rf_model3.predict(x_test)

In [ ]:
accuracy_score(y_test,y_pred_rf3)

In [ ]:
cm_dt=confusion_matrix(y_test,y_pred_rf3)

plt.figure(figsize=(10,7))
sns.heatmap(cm_dt,annot=True,fmt='d',cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

print(classification_report(y_test,y_pred_rf3))

In [ ]:
dt_model4= DecisionTreeClassifier()
dt_model4.fit(x_train,y_train)

In [ ]:
y_pred_rf4=dt_model4.predict(x_test)

In [ ]:
accuracy_score(y_test,y_pred_rf4)

In [ ]:
from numpy.random.mtrand import gamma
model5=SVC(kernel='rbf', C=1.0,gamma='scale')
model5.fit(x_train,y_train)

In [ ]:
y_pred_rf5=model5.predict(x_test)

In [ ]:
accuracy_score(y_test,y_pred_rf5)

In [ ]:
k=16
model = KNeighborsClassifier(n_neighbors=k)
model.fit(x_train, y_train)

In [ ]:
y_pred = model.predict(x_test)

In [ ]:
y_pred

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

In [ ]:
conf_matrix = confusion_matrix(y_test, y_pred)
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels= ['Not effective','Effective'],
            yticklabels= ['Not effective','Effective'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
accuracy_scores =[]
k_values = range(1,20)

for k in k_values:
  model = KNeighborsClassifier(n_neighbors=k)
  model.fit(x_train, y_train)
  y_pred = model.predict(x_test)
  accuracy_scores.append(accuracy_score(y_test, y_pred))


plt.plot(k_values, accuracy_scores, marker='o')
plt.xlabel('Number of Neighbors(K)')
plt.ylabel('Accuracy')
plt.title('Accuracy vs Number of Neighbors(K)')
plt.show()

In [ ]:
param_grid={
    'n_estimators':[50,100,150],
    'max_depth':[None,10,20],
    'min_samples_split':[2,5,10],
    'min_samples_leaf':[1,2,4]
}

In [ ]:
grid_search=GridSearchCV(RandomForestClassifier(random_state=42),param_grid,cv=5,scoring='accuracy')
grid_search.fit(x_train,y_train)
print('Best Hyperparameters for Random Forest:', grid_search.best_params_)

In [ ]:
y_pred_best_rf=grid_search.best_estimator_.predict(x_test)
print('Tuned Random forest accuracy:',accuracy_score(y_test,y_pred_best_rf))

In [ ]:
import joblib
best_model=grid_search.best_estimator_
joblib.dump(best_model,'tuned_random_forest_model.joblib')
print("Model saved as 'tuned_random_forest_model.joblib'")


In [ ]:
import gradio as gr
import pandas as pd
import numpy as np

# Ensure loaded_model, scaler, and x (original dataframe for column alignment) are available
# These objects were created and are available in the current kernel state.

def predict_churn(credit_score, geography, gender, age, tenure, balance, num_of_products, has_cr_card, is_active_member, estimated_salary):
    # Create a dictionary for the new data point
    data = {
        'CreditScore': credit_score,
        'Gender': gender, # Will be mapped below
        'Age': age,
        'Tenure': tenure,
        'Balance': balance,
        'NumOfProducts': num_of_products,
        'HasCrCard': has_cr_card,
        'IsActiveMember': is_active_member,
        'EstimatedSalary': estimated_salary,
        # Initialize one-hot encoded geography columns
        'Geography_Germany': 0,
        'Geography_Spain': 0
    }

    # Create a DataFrame from the single data point
    sample_df = pd.DataFrame([data])

    # Apply preprocessing steps similar to training
    # 1. Encode Gender (Binary Encoding)
    sample_df['Gender'] = sample_df['Gender'].map({'Male': 0, 'Female': 1})

    # 2. One-Hot Encode Geography
    if geography == 'Germany':
        sample_df['Geography_Germany'] = 1
    elif geography == 'Spain':
        sample_df['Geography_Spain'] = 1
    # 'France' implies both 'Geography_Germany' and 'Geography_Spain' remain 0

    # Ensure the order of columns matches the training data (x.columns)
    # x is a DataFrame in the kernel state that holds the preprocessed features before scaling.
    # Its columns represent the expected input features for the model.
    final_features_df = sample_df.reindex(columns=x.columns, fill_value=0)

    # 3. Scale numerical features using the pre-fitted scaler
    scaled_input = scaler.transform(final_features_df)

    # 4. Make prediction
    prediction = loaded_model.predict(scaled_input)[0]
    prediction_proba = loaded_model.predict_proba(scaled_input)[0]

    # 5. Return human-readable result
    if prediction == 1:
        return f"Prediction: Customer will churn (Probability: {prediction_proba[1]:.2f})"
    else:
        return f"Prediction: Customer will not churn (Probability: {prediction_proba[0]:.2f})"

# Define Gradio input components
credit_score_input = gr.Slider(minimum=350, maximum=850, step=1, value=650, label='Credit Score')
geography_input = gr.Dropdown(choices=['France', 'Germany', 'Spain'], value='France', label='Geography')
gender_input = gr.Radio(choices=['Male', 'Female'], value='Male', label='Gender')
age_input = gr.Slider(minimum=18, maximum=92, step=1, value=35, label='Age') # Max age based on common demographics for dataset types
tenure_input = gr.Slider(minimum=0, maximum=10, step=1, value=5, label='Tenure (years)')
balance_input = gr.Slider(minimum=0.0, maximum=250000.0, step=100.0, value=80000.0, label='Balance') # Max balance based on typical dataset ranges
num_of_products_input = gr.Slider(minimum=1, maximum=4, step=1, value=1, label='Number of Products')
has_cr_card_input = gr.Radio(choices=[0, 1], value=1, label='Has Credit Card (0=No, 1=Yes)')
is_active_member_input = gr.Radio(choices=[0, 1], value=1, label='Is Active Member (0=No, 1=Yes)')
estimated_salary_input = gr.Slider(minimum=0.0, maximum=200000.0, step=100.0, value=95000.0, label='Estimated Salary') # Max salary based on typical dataset ranges

# Create the Gradio interface
interface = gr.Interface(
    fn=predict_churn,
    inputs=[
        credit_score_input,
        geography_input,
        gender_input,
        age_input,
        tenure_input,
        balance_input,
        num_of_products_input,
        has_cr_card_input,
        is_active_member_input,
        estimated_salary_input
    ],
    outputs='text',
    title='Customer Churn Prediction',
    description='Enter customer details to predict if they will churn (exit the bank).'
)

# Launch the interface
interface.launch()